# Fabric vs. Databricks — Side-by-Side Comparison
RetailBank: 50K transactions, same logic, two implementations.

In [ ]:
import subprocess, sys
for pkg in ['duckdb','faker','numpy']:
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True)

## Run both pipelines

In [ ]:
import os; os.chdir('..')
from fabric_approach import run as fabric_run
fabric_run()

In [ ]:
from databricks_approach import run as db_run
db_run()

## Compare outputs

In [ ]:
import pandas as pd, duckdb
fabric_gold = duckdb.connect('data/fabric_bank.duckdb').execute(
    'SELECT merchant_category, SUM(total_transactions) AS txn, '
    'ROUND(SUM(total_volume_gbp)/1e6,2) AS vol_M FROM gold_daily_risk '
    'GROUP BY merchant_category ORDER BY vol_M DESC'
).df()
print('FABRIC OUTPUT:')
print(fabric_gold.to_string(index=False))

In [ ]:
db_gold = pd.read_parquet('data/databricks_bank/gold/daily_risk.parquet')
db_summary = (db_gold.groupby('merchant_category')
    .agg(txn=('total_transactions','sum'), vol_M=('total_volume_gbp', lambda x: round(x.sum()/1e6,2)))
    .sort_values('vol_M', ascending=False))
print('DATABRICKS OUTPUT:')
print(db_summary.to_string())

## Key differences in implementation

| Aspect | Fabric | Databricks |
|---|---|---|
| Write semantics | TRUNCATE + INSERT | MERGE (upsert) |
| Schema evolution | SQL ALTER TABLE | `mergeSchema=true` |
| Incremental update | Full reload | Delta MERGE |
| Table format | DuckDB (→ Delta) | Delta native |

## Conclusion
Same business output. Different engineering approach. See `decision_framework.md` for when to choose each.